# Talent Knowledge Graph Builder

Notebook ini menyatukan berbagai sumber data talenta UNESA dan memuatnya ke dalam Neo4j sebagai knowledge graph yang kaya serta terintegrasi. Prosesnya mencakup pemuatan data, normalisasi, pembuatan node dan relasi, hingga analitik berbasis Graph Data Science.


In [ ]:
# ===== Standard Libraries =====
import os
import ast
import json
import math
import hashlib
import re
from pathlib import Path
from typing import Dict, Iterable, List, Optional

import numpy as np
import pandas as pd
from neo4j import GraphDatabase, RoutingControl
from neo4j.exceptions import Neo4jError, ServiceUnavailable

try:
    from graphdatascience import GraphDataScience
except ImportError:  # pragma: no cover
    GraphDataScience = None

from dotenv import load_dotenv
from IPython.display import display

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)


def split_dataframe(df: pd.DataFrame, chunk_size: int = 1_000) -> Iterable[pd.DataFrame]:
    for start in range(0, len(df), chunk_size):
        yield df.iloc[start : start + chunk_size]


def clean_records(df: pd.DataFrame) -> List[Dict[str, object]]:
    return df.replace({np.nan: None}).to_dict(orient="records")


def slugify(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9]+", "-", text)
    return text.strip("-") or "unknown"


In [ ]:
# 1) Load environment variables dan tentukan lokasi dataset
for env_file in ("ws.env", ".env"):
    if Path(env_file).exists():
        load_dotenv(env_file, override=True)

NEO4J_URI = os.getenv("NEO4J_URI", "neo4j://localhost:7687")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME", os.getenv("NEO4J_USER", "neo4j"))
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "neo4j")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")


def find_dataset_directory() -> Path:
    current = Path.cwd().resolve()
    for base in [current, *current.parents]:
        talent_dir = base / "notebook" / "build-graph" / "talent"
        if (talent_dir / "penelitian_with_research_areas.csv").exists():
            return talent_dir
        if (base / "penelitian_with_research_areas.csv").exists():
            return base
    raise FileNotFoundError("Tidak dapat menemukan dataset talent.")


TALENT_DIR = find_dataset_directory()
print(f"📂 Dataset directory: {TALENT_DIR}")

# 2) Helper parsing utilities
def parse_literal(value):
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return None
    if isinstance(value, (list, dict, tuple, set)):
        return value
    text = str(value).strip()
    if not text:
        return None
    try:
        return ast.literal_eval(text)
    except (ValueError, SyntaxError):
        return None


def ensure_list(value) -> List[str]:
    parsed = parse_literal(value)
    if parsed is None:
        if value is None:
            return []
        return [str(value).strip()] if str(value).strip() else []
    if isinstance(parsed, dict):
        return [str(k) for k in parsed.keys()]
    return [str(item).strip() for item in list(parsed) if str(item).strip()]


def ensure_dict(value) -> Dict[str, float]:
    parsed = parse_literal(value)
    if isinstance(parsed, dict):
        return {str(k): float(v) for k, v in parsed.items()}
    return {}


# 3) Load dataset utama
DATA_FILES = {
    "penelitian": "penelitian_with_research_areas.csv",
    "summary": "dosen_expertise_summary.csv",
    "analysis": "dosen_expertise_analysis.csv",
}

RAW_DATA: Dict[str, pd.DataFrame] = {}
for key, filename in DATA_FILES.items():
    path = TALENT_DIR / filename
    if not path.exists():
        print(f"⚠️  File {filename} tidak ditemukan")
        RAW_DATA[key] = pd.DataFrame()
        continue
    RAW_DATA[key] = pd.read_csv(path)
    print(f"✅ Loaded {key:>8}: {RAW_DATA[key].shape[0]:,} rows × {RAW_DATA[key].shape[1]} columns")

penelitian_df = RAW_DATA["penelitian"].copy()
summary_df = RAW_DATA["summary"].copy()
analysis_df = RAW_DATA["analysis"].copy()

if penelitian_df.empty:
    raise ValueError("Dataset penelitian kosong sehingga graf tidak dapat dibangun.")

# 4) Normalisasi data penelitian
penelitian_df["tahun_kegiatan"] = pd.to_numeric(penelitian_df["tahun_kegiatan"], errors="coerce").astype("Int64")
penelitian_df["research_area_list"] = penelitian_df["research_areas"].apply(ensure_list)
penelitian_df["research_area_count"] = penelitian_df["research_area_list"].apply(len)
penelitian_df["jenis_kegiatan"] = penelitian_df["jenis_kegiatan"].fillna("Penelitian")
penelitian_df["project_id"] = penelitian_df.apply(
    lambda row: "PROJECT::" + hashlib.sha1(f"{row['nama_dosen']}|{row['judul_kegiatan']}|{row['tahun_kegiatan']}".encode("utf-8")).hexdigest(),
    axis=1,
)
penelitian_df["dosen_id"] = penelitian_df.apply(
    lambda row: str(row["id_sdm"]) if pd.notna(row["id_sdm"]) else f"NAME::{slugify(row['nama_dosen'])}",
    axis=1,
)
penelitian_df["judul_kegiatan"] = penelitian_df["judul_kegiatan"].str.strip()

# 5) Integrasi ringkasan keahlian
def prepare_dosen_profiles() -> pd.DataFrame:
    summary = summary_df.rename(
        columns={
            "Nama Dosen": "nama_dosen",
            "Total Penelitian": "total_penelitian",
            "Periode": "periode_summary",
            "Primary Expertise": "primary_expertise_summary",
            "Secondary Areas": "secondary_areas_summary",
            "Diversity Score": "diversity_score_summary",
        }
    )
    summary["secondary_areas_summary"] = summary["secondary_areas_summary"].apply(ensure_list)

    analysis = analysis_df.rename(
        columns={
            "nama_dosen": "nama_dosen",
            "jumlah_penelitian": "jumlah_penelitian_analysis",
            "periode_penelitian": "periode_analysis",
            "primary_expertise": "primary_expertise_analysis",
            "secondary_expertise": "secondary_expertise_analysis",
            "all_areas": "all_areas_analysis",
            "area_distribution": "area_distribution_analysis",
            "expertise_diversity": "diversity_score_analysis",
        }
    )
    analysis["secondary_expertise_analysis"] = analysis["secondary_expertise_analysis"].apply(ensure_list)
    analysis["all_areas_analysis"] = analysis["all_areas_analysis"].apply(ensure_list)
    analysis["area_distribution_analysis"] = analysis["area_distribution_analysis"].apply(ensure_dict)
    analysis["periode_analysis"] = analysis["periode_analysis"].fillna("")

    base = penelitian_df[["dosen_id", "id_sdm", "nama_dosen"]].drop_duplicates().reset_index(drop=True)
    base = base.merge(summary, on="nama_dosen", how="left")
    base = base.merge(analysis, on="nama_dosen", how="left")

    activity = (
        penelitian_df.groupby(["dosen_id", "nama_dosen"], as_index=False)
        .agg(
            project_count=("project_id", "nunique"),
            latest_activity_year=("tahun_kegiatan", "max"),
            earliest_activity_year=("tahun_kegiatan", "min"),
        )
    )
    base = base.merge(activity, on=["dosen_id", "nama_dosen"], how="left")

    def choose_primary(row):
        if isinstance(row.get("primary_expertise_analysis"), str) and row["primary_expertise_analysis"].strip():
            return row["primary_expertise_analysis"], "analysis"
        if isinstance(row.get("primary_expertise_summary"), str) and row["primary_expertise_summary"].strip():
            return row["primary_expertise_summary"], "summary"
        return None, None

    primary_vals = base.apply(choose_primary, axis=1, result_type="expand")
    base["primary_expertise"] = primary_vals[0]
    base["primary_source"] = primary_vals[1]

    def build_secondary(row) -> List[str]:
        buckets: List[str] = []
        for col in ("secondary_areas_summary", "secondary_expertise_analysis", "all_areas_analysis"):
            values = row.get(col)
            if isinstance(values, list):
                buckets.extend(values)
        primary = row.get("primary_expertise")
        return sorted({area for area in buckets if area and area != primary})

    base["secondary_areas"] = base.apply(build_secondary, axis=1)
    base["area_distribution"] = base.get("area_distribution_analysis", pd.Series([{}] * len(base)))

    def combine_periods(row) -> List[str]:
        buckets: List[str] = []
        for col in ("periode_summary", "periode_analysis"):
            value = row.get(col)
            if isinstance(value, str) and value.strip():
                buckets.extend([p.strip() for p in value.split(",") if p.strip()])
        start, end = row.get("earliest_activity_year"), row.get("latest_activity_year")
        if pd.notna(start) and pd.notna(end):
            buckets.append(f"{int(start)}-{int(end)}")
        return sorted(set(buckets))

    base["periods"] = base.apply(combine_periods, axis=1)

    def choose_diversity(row) -> Optional[float]:
        for col in ("diversity_score_analysis", "diversity_score_summary"):
            value = row.get(col)
            if pd.notna(value):
                return float(value)
        return None

    base["diversity_score"] = base.apply(choose_diversity, axis=1)
    base["project_count"] = base["project_count"].fillna(base["total_penelitian"])
    return base


dosen_nodes = prepare_dosen_profiles()

# 6) Derivasi node & relasi
project_nodes = (
    penelitian_df[["project_id", "judul_kegiatan", "tahun_kegiatan", "jenis_kegiatan", "research_area_list"]]
    .drop_duplicates("project_id")
    .reset_index(drop=True)
)
project_nodes["research_area_count"] = project_nodes["research_area_list"].apply(len)

dosen_project_edges = penelitian_df[["dosen_id", "project_id", "tahun_kegiatan", "jenis_kegiatan"]].drop_duplicates()

project_area_edges = (
    penelitian_df[["project_id", "tahun_kegiatan", "research_area_list"]]
    .explode("research_area_list")
    .dropna(subset=["research_area_list"])
    .rename(columns={"research_area_list": "research_area"})
)

dosen_research_edges = (
    penelitian_df[["dosen_id", "nama_dosen", "project_id", "tahun_kegiatan", "research_area_list"]]
    .explode("research_area_list")
    .dropna(subset=["research_area_list"])
    .rename(columns={"research_area_list": "research_area"})
)

dosen_research_summary = (
    dosen_research_edges.groupby(["dosen_id", "nama_dosen", "research_area"], as_index=False)
    .agg(
        project_count=("project_id", "nunique"),
        first_year=("tahun_kegiatan", "min"),
        last_year=("tahun_kegiatan", "max"),
    )
)

research_area_nodes = (
    dosen_research_edges.groupby("research_area", as_index=False)
    .agg(
        project_count=("project_id", "nunique"),
        dosen_count=("dosen_id", "nunique"),
        latest_year=("tahun_kegiatan", "max"),
    )
)

primary_counts = (
    dosen_nodes.dropna(subset=["primary_expertise"])\
    .groupby("primary_expertise")\
    .agg(primary_count=("dosen_id", "nunique"))\
    .rename_axis("research_area")\
    .reset_index()
)

secondary_counts = (
    dosen_nodes.explode("secondary_areas")\
    .dropna(subset=["secondary_areas"])\
    .groupby("secondary_areas")\
    .agg(secondary_count=("dosen_id", "nunique"))\
    .rename_axis("research_area")\
    .reset_index()
)

research_area_nodes = research_area_nodes.merge(primary_counts, on="research_area", how="left")
research_area_nodes = research_area_nodes.merge(secondary_counts, on="research_area", how="left")
research_area_nodes["primary_count"] = research_area_nodes["primary_count"].fillna(0).astype(int)
research_area_nodes["secondary_count"] = research_area_nodes["secondary_count"].fillna(0).astype(int)

# primary & secondary expertise edges
primary_edges = (
    dosen_nodes.dropna(subset=["primary_expertise"])
    [["dosen_id", "primary_expertise", "primary_source"]]
    .rename(columns={"primary_expertise": "research_area"})
)

secondary_edges = (
    dosen_nodes[["dosen_id", "secondary_areas"]]
    .explode("secondary_areas")
    .dropna(subset=["secondary_areas"])
    .rename(columns={"secondary_areas": "research_area"})
)
secondary_edges["rank"] = secondary_edges.groupby("dosen_id").cumcount() + 1

area_distribution_edges: List[Dict[str, object]] = []
for row in dosen_nodes.itertuples():
    distribution = row.area_distribution or {}
    for area, weight in distribution.items():
        area_distribution_edges.append({"dosen_id": row.dosen_id, "research_area": area, "weight": float(weight)})
area_distribution_edges = pd.DataFrame(area_distribution_edges)

print(
    f"Nodes — Dosen: {len(dosen_nodes):,}, Project: {len(project_nodes):,}, ResearchArea: {len(research_area_nodes):,}"
)
print(
    f"Relationships — CONDUCTED: {len(dosen_project_edges):,}, FOCUSES_ON: {len(project_area_edges):,}, RESEARCHES: {len(dosen_research_summary):,}"
)


In [ ]:
# 7) Koneksi ke Neo4j dan pembuatan constraint
NEO4J_AVAILABLE = False
driver = None
gds = None
try:
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
    driver.verify_connectivity()
    NEO4J_AVAILABLE = True
    print("✅ Connected to Neo4j")
except (Neo4jError, ServiceUnavailable) as exc:
    print(f"⚠️  Neo4j tidak tersedia: {exc}")

if NEO4J_AVAILABLE and GraphDataScience is not None:
    try:
        gds = GraphDataScience.from_neo4j_driver(driver)
        print("✅ Graph Data Science client siap")
    except Exception as exc:
        print(f"⚠️  Graph Data Science client gagal diinisialisasi: {exc}")
        gds = None

if NEO4J_AVAILABLE:
    constraint_queries = [
        "CREATE CONSTRAINT IF NOT EXISTS FOR (d:Dosen) REQUIRE d.dosen_id IS UNIQUE",
        "CREATE CONSTRAINT IF NOT EXISTS FOR (p:Project) REQUIRE p.project_id IS UNIQUE",
        "CREATE CONSTRAINT IF NOT EXISTS FOR (ra:ResearchArea) REQUIRE ra.name IS UNIQUE",
    ]
    for query in constraint_queries:
        driver.execute_query(query, database_=NEO4J_DATABASE, routing_=RoutingControl.WRITE)
    print("✅ Constraints ensured")
else:
    print("➡️  Lewati pembuatan constraint dan loading data")


In [ ]:
if NEO4J_AVAILABLE:
    def run_write(query: str, df: pd.DataFrame) -> None:
        for chunk in split_dataframe(df):
            driver.execute_query(
                query,
                rows=clean_records(chunk),
                database_=NEO4J_DATABASE,
                routing_=RoutingControl.WRITE,
            )

    dosen_query = """
    UNWIND $rows AS row
    MERGE (d:Dosen {dosen_id: row.dosen_id})
    SET d.name = row.nama_dosen,
        d.id_sdm = coalesce(row.id_sdm, d.id_sdm),
        d.project_count = coalesce(row.project_count, d.project_count),
        d.total_penelitian = coalesce(row.total_penelitian, d.total_penelitian),
        d.primary_expertise = coalesce(row.primary_expertise, d.primary_expertise),
        d.secondary_areas = coalesce(row.secondary_areas, d.secondary_areas),
        d.area_distribution = coalesce(row.area_distribution, d.area_distribution),
        d.diversity_score = coalesce(row.diversity_score, d.diversity_score),
        d.periods = coalesce(row.periods, d.periods),
        d.primary_source = coalesce(row.primary_source, d.primary_source),
        d.latest_activity_year = coalesce(row.latest_activity_year, d.latest_activity_year),
        d.earliest_activity_year = coalesce(row.earliest_activity_year, d.earliest_activity_year)
    """
    run_write(
        dosen_query,
        dosen_nodes[[
            "dosen_id",
            "nama_dosen",
            "id_sdm",
            "project_count",
            "total_penelitian",
            "primary_expertise",
            "secondary_areas",
            "area_distribution",
            "diversity_score",
            "periods",
            "primary_source",
            "latest_activity_year",
            "earliest_activity_year",
        ]],
    )

    project_query = """
    UNWIND $rows AS row
    MERGE (p:Project {project_id: row.project_id})
    SET p.title = row.judul_kegiatan,
        p.year = coalesce(row.tahun_kegiatan, p.year),
        p.activity_type = coalesce(row.jenis_kegiatan, p.activity_type),
        p.research_areas = coalesce(row.research_area_list, p.research_areas),
        p.research_area_count = coalesce(row.research_area_count, p.research_area_count)
    """
    run_write(project_query, project_nodes)

    research_area_query = """
    UNWIND $rows AS row
    MERGE (ra:ResearchArea {name: row.research_area})
    SET ra.project_count = coalesce(row.project_count, ra.project_count),
        ra.dosen_count = coalesce(row.dosen_count, ra.dosen_count),
        ra.latest_year = coalesce(row.latest_year, ra.latest_year),
        ra.primary_count = coalesce(row.primary_count, ra.primary_count),
        ra.secondary_count = coalesce(row.secondary_count, ra.secondary_count)
    """
    run_write(research_area_query, research_area_nodes)

    dosen_project_query = """
    UNWIND $rows AS row
    MATCH (d:Dosen {dosen_id: row.dosen_id})
    MATCH (p:Project {project_id: row.project_id})
    MERGE (d)-[r:CONDUCTED]->(p)
    SET r.year = coalesce(row.tahun_kegiatan, r.year),
        r.activity_type = coalesce(row.jenis_kegiatan, r.activity_type)
    """
    run_write(dosen_project_query, dosen_project_edges)

    project_area_query = """
    UNWIND $rows AS row
    MATCH (p:Project {project_id: row.project_id})
    MATCH (ra:ResearchArea {name: row.research_area})
    MERGE (p)-[r:FOCUSES_ON]->(ra)
    SET r.year = coalesce(row.tahun_kegiatan, r.year)
    """
    run_write(project_area_query, project_area_edges)

    research_relation_query = """
    UNWIND $rows AS row
    MATCH (d:Dosen {dosen_id: row.dosen_id})
    MATCH (ra:ResearchArea {name: row.research_area})
    MERGE (d)-[r:RESEARCHES]->(ra)
    SET r.project_count = coalesce(row.project_count, r.project_count, 0),
        r.first_year = coalesce(row.first_year, r.first_year),
        r.last_year = coalesce(row.last_year, r.last_year)
    """
    run_write(
        research_relation_query,
        dosen_research_summary[["dosen_id", "research_area", "project_count", "first_year", "last_year"]],
    )

    primary_relation_query = """
    UNWIND $rows AS row
    MATCH (d:Dosen {dosen_id: row.dosen_id})
    MATCH (ra:ResearchArea {name: row.research_area})
    MERGE (d)-[r:PRIMARY_EXPERTISE]->(ra)
    SET r.source = coalesce(row.primary_source, r.source)
    """
    run_write(primary_relation_query, primary_edges)

    secondary_relation_query = """
    UNWIND $rows AS row
    MATCH (d:Dosen {dosen_id: row.dosen_id})
    MATCH (ra:ResearchArea {name: row.research_area})
    MERGE (d)-[r:SECONDARY_EXPERTISE]->(ra)
    SET r.rank = coalesce(row.rank, r.rank)
    """
    run_write(secondary_relation_query, secondary_edges[["dosen_id", "research_area", "rank"]])

    if not area_distribution_edges.empty:
        weighted_relation_query = """
        UNWIND $rows AS row
        MATCH (d:Dosen {dosen_id: row.dosen_id})
        MATCH (ra:ResearchArea {name: row.research_area})
        MERGE (d)-[r:AREA_FOCUS]->(ra)
        SET r.weight = coalesce(row.weight, r.weight)
        """
        run_write(weighted_relation_query, area_distribution_edges)

    print("✅ Nodes & relationships loaded ke Neo4j")


In [ ]:
if NEO4J_AVAILABLE:
    top_research_area_df = driver.execute_query(
        """
        MATCH (d:Dosen)-[r:RESEARCHES]->(ra:ResearchArea)
        RETURN ra.name AS research_area,
               COUNT(DISTINCT d) AS dosen_count,
               SUM(r.project_count) AS project_mentions
        ORDER BY project_mentions DESC
        LIMIT 10
        """,
        database_=NEO4J_DATABASE,
        routing_=RoutingControl.READ,
        result_transformer_=lambda r: r.to_df(),
    )
    display(top_research_area_df)
else:
    print("⚠️  Neo4j offline - lewati eksplorasi graf")

if NEO4J_AVAILABLE and gds is not None:
    graph_name = "talent_dosen_similarity"
    try:
        gds.run_cypher(f"CALL gds.graph.drop('{graph_name}', false)")
    except Exception:
        pass

    node_config = {"Dosen": {"properties": ["project_count", "diversity_score"]}}
    rel_config = {"RESEARCHES": {"orientation": "UNDIRECTED", "properties": ["project_count"]}}

    G, _ = gds.graph.project(graph_name, node_config, rel_config)

    similarity_df = gds.nodeSimilarity.stream(
        graph_name,
        relationshipWeightProperty="project_count",
        topK=5,
    ).to_pandas()

    communities_df = gds.leiden.stream(
        graph_name,
        relationshipWeightProperty="project_count",
        includeIntermediateCommunities=False,
    ).to_pandas()

    gds.graph.drop(graph_name)

    print("Similarity edges sample:")
    display(similarity_df.head())
    print("\nCommunity sample:")
    display(communities_df.head())
else:
    print("➡️  Graph Data Science tidak tersedia atau Neo4j offline")


Relasi Jadwal (DIJADWALKAN_PADA & MEMILIKI_JADWAL) telah dibuat tanpa duplikasi


In [ ]:
summary = {
    "dosen_nodes": len(dosen_nodes),
    "project_nodes": len(project_nodes),
    "research_area_nodes": len(research_area_nodes),
    "dosen_project_edges": len(dosen_project_edges),
    "project_area_edges": len(project_area_edges),
    "dosen_research_edges": len(dosen_research_summary),
}
print(json.dumps(summary, indent=2))
